In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
print("TensorFlow version:", tf.__version__)

In [ ]:
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [7]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
 import zipfile
import os
zip_file = list(uploaded.keys())[0]
extract_path = "/content/intel_dataset"
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Dataset extracted successfully!")

In [ ]:
# Check extracted folders
for root, dirs, files in os.walk(extract_path):
    print(root)
    print("Folders:", dirs[:10])
    print("Files:", len(files))
    print("-" * 50)
    if root.count(os.sep) > extract_path.count(os.sep) + 2:
        dirs[:] = []

In [ ]:
# Find train and test folders
train_path = None
test_path = None
pred_path = None
for root, dirs, files in os.walk(extract_path):

    if "seg_train" in dirs:
        train_path = os.path.join(root, "seg_train")

    if "seg_test" in dirs:
        test_path = os.path.join(root, "seg_test")

    if "seg_pred" in dirs:
        pred_path = os.path.join(root, "seg_pred")

print("Train path:", train_path)
print("Test path:", test_path)
print("Prediction path:", pred_path)

In [ ]:
classes = sorted(os.listdir(train_path))
print("Classes:")
for i, class_name in enumerate(classes):
    print(i, ":", class_name)

print("\nNumber of classes:", len(classes))

In [ ]:
# Count images in each class

class_counts = {}
for class_name in classes:
    class_folder = os.path.join(train_path, class_name)

    image_count = len([
        file for file in os.listdir(class_folder)
        if file.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    class_counts[class_name] = image_count
class_counts

In [ ]:
# Convert to DataFrame
class_df = pd.DataFrame(
    list(class_counts.items()),
    columns=["Class", "Number of Images"]
)
class_df

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(
    class_df["Class"],
    class_df["Number of Images"]
)
plt.title("Number of Training Images per Class")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Display one sample image from each class

plt.figure(figsize=(15, 8))
for i, class_name in enumerate(classes):

    class_folder = os.path.join(train_path, class_name)
    image_files = [
        file for file in os.listdir(class_folder)
        if file.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    image_path = os.path.join(class_folder, image_files[0])

    image = plt.imread(image_path)
    plt.subplot(2, 3, i + 1)
    plt.imshow(image)
    plt.title(class_name)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image
sample_class = classes[0]
sample_folder = os.path.join(train_path, sample_class)
sample_image_file = [
    file for file in os.listdir(sample_folder)
    if file.lower().endswith(('.jpg', '.jpeg', '.png'))
][0]

sample_image_path = os.path.join(
    sample_folder,
    sample_image_file
)

image = Image.open(sample_image_path)
print("Image size:", image.size)
print("Image mode:", image.mode)

In [ ]:
# Image settings

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)

In [ ]:
# Create training dataset

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

In [ ]:
# Create validation dataset

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

In [ ]:
print("Class names:", train_dataset.class_names)

In [ ]:
# Create test dataset
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)
print("Test classes:", test_dataset.class_names)

In [ ]:
# Data augmentation
data_augmentation = tf.keras.Sequential([

    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(
        height_factor=0.1,
        width_factor=0.1
    )

])
print("Data augmentation created successfully.")

In [ ]:
# Display augmented images
for images, labels in train_dataset.take(1):
    first_image = images[0]
    plt.figure(figsize=(12, 8))
    for i in range(6):
        augmented_image = data_augmentation(
            tf.expand_dims(first_image, 0),
            training=True
        )[0]
        plt.subplot(2, 3, i + 1)
        plt.imshow(augmented_image.numpy().astype("uint8"))
        plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(
    buffer_size=AUTOTUNE
)
validation_dataset = validation_dataset.prefetch(
    buffer_size=AUTOTUNE
)
test_dataset = test_dataset.prefetch(
    buffer_size=AUTOTUNE
)

In [ ]:
# Build CNN model
model = models.Sequential([

    # Data augmentation
    data_augmentation,

    # Normalize pixel values
    layers.Rescaling(1./255),

    # Convolution Block 1
    layers.Conv2D(
        32,
        (3, 3),
        activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    # Convolution Block 2
    layers.Conv2D(
        64,
        (3, 3),
        activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    # Convolution Block 3
    layers.Conv2D(
        128,
        (3, 3),
        activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    # Convolution Block 4
    layers.Conv2D(
        128,
        (3, 3),
        activation="relu",
        padding="same"
    ),
    layers.MaxPooling2D((2, 2)),

    # Flatten
    layers.Flatten(),

    # Fully connected layer
    layers.Dense(
        128,
        activation="relu"
    ),

    # Dropout
    layers.Dropout(0.5),

    # Output layer
    layers.Dense(
        6,
        activation="softmax"
    )
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# Train the model
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20

)